# Chapter 4: Training Fundamentals

## 100 Bad Guessers That Become One Great Predictor

We now have a clean numeric matrix (from Chapter 3). Time to teach a model to predict churn.

But before we hand data to XGBoost, we need to handle two critical setup problems:

1. **How do we split data fairly?** — The model needs a textbook (training), practice exams (validation), and a final exam (test). Each group must have the same proportion of churners as the original.

2. **What do we do about imbalance?** — If only 15% of customers churn, a lazy model can predict "won't churn" for everyone and be right 85% of the time. That's useless.

This chapter covers both problems, plus how we interpret predictions with risk tiers and SHAP explanations.

In [ ]:
import numpy as np
import pandas as pd
from churn_pipeline.steps.training import (
    create_stratified_splits,
    compute_scale_pos_weight,
    HYPERPARAMETER_RANGES,
)
from churn_pipeline.steps.scoring import (
    assign_risk_tier,
    extract_top_reasons,
    format_predictions,
)

## Part 1: Stratified Splitting

### The Problem

Imagine a bag with 80 blue marbles and 20 red marbles. If you grab a handful randomly, you might get 15 blue and 0 red — that handful doesn't represent the bag.

Same with customer data. If 20% of customers churned overall, each split (train/validation/test) should have ~20% churners. Otherwise:
- Training on 30% churners, testing on 10% churners → performance estimate is unreliable
- The model sees a different "world" in each split

**Stratified splitting** guarantees each group looks like the original.

In [ ]:
# Create a realistic dataset: 1000 customers, 20% churned
np.random.seed(42)
n_customers = 1000
n_features = 6

features = np.random.rand(n_customers, n_features)
labels = np.array([0] * 800 + [1] * 200)  # 20% churn rate
np.random.shuffle(labels)

print(f"Full dataset: {n_customers} customers, {labels.mean():.1%} churn rate")
print(f"Feature matrix shape: {features.shape}")
print(f"  → {features.shape[0]} rows (one per customer)")
print(f"  → {features.shape[1]} columns (one per feature, e.g. tenure, charges, etc.)")
print(f"\nLabels: 0 = stayed, 1 = left")
print(f"  {800} customers stayed (label=0)")
print(f"  {200} customers left (label=1)")

In [ ]:
# Split into train (70%), validation (15%), test (15%)
(X_train, y_train), (X_val, y_val), (X_test, y_test) = create_stratified_splits(
    features, labels
)

print(f"{'Split':<12} {'Size':<8} {'Churn Rate':<12} {'Preserved?'}")
print("-" * 45)
print(f"{'Original':<12} {n_customers:<8} {labels.mean():.3f}{'':>8}")
print(f"{'Train':<12} {len(y_train):<8} {y_train.mean():.3f}{'':>8} ✓")
print(f"{'Validation':<12} {len(y_val):<8} {y_val.mean():.3f}{'':>8} ✓")
print(f"{'Test':<12} {len(y_test):<8} {y_test.mean():.3f}{'':>8} ✓")
print(f"\nTotal samples: {len(y_train) + len(y_val) + len(y_test)} (no data lost)")
print(f"\nWhat to notice:")
print(f"  • The churn rate is 0.200 (20%) in EVERY group — not just overall.")
print(f"  • This means Train, Validation, and Test all 'look like' the real world.")
print(f"  • If we hadn't done this, one group might have 30% churners and another")
print(f"    might have 10% — the model would be trained on a distorted reality.")

### Why 70/15/15?

| Split | Purpose | Analogy |
|-------|---------|--------|
| Train (70%) | The model reads this to learn patterns | The textbook |
| Validation (15%) | Checked during training to prevent memorization | Practice exams |
| Test (15%) | Final honest evaluation — never seen during training | The final exam |

The validation set is how we know when to stop training. Without it, the model would memorize every customer in the training set ("overfitting") instead of learning general patterns.

## Part 2: Handling Class Imbalance

### The Lazy Model Problem

If 85% of customers stay, a model can achieve 85% accuracy by always predicting "won't churn." That's useless — it catches zero actual leavers.

To understand the fix, we first need to understand how a model learns.

### How a Model Learns: The Score Card

When the model makes a prediction, it can be right or wrong. After each round of predictions, we calculate a **score** that measures "how badly did the model do?" — the worse the predictions, the higher this score. Think of it like golf: lower is better.

This score is called a **loss function** — it's literally "how much did we lose by being wrong?" The model's entire goal during training is to make this score as low as possible. Each new tree it builds is specifically designed to reduce this score.

### The Fix: Making Mistakes on Churners Cost More

Here's the trick: by default, getting a churner wrong costs the same as getting a stayer wrong on that score card. But churners are rare (only 15%), so the model barely notices them.

`scale_pos_weight` changes the scoring rules. It says: "Every time you get a churner wrong, it counts as N mistakes instead of 1." Now missing a churner HURTS the score much more than missing a stayer, so the model works harder to get churners right.

Formula: `scale_pos_weight = count(stayers) / count(churners)`

If 850 stayed and 150 left: weight = 850/150 = 5.67 → each missed churner hurts 5.67× more than a missed stayer.

In [ ]:
# With our 80/20 split
weight = compute_scale_pos_weight(labels)
print(f"Dataset: {800} stayed, {200} left")
print(f"scale_pos_weight = {800}/{200} = {weight:.2f}")
print(f"\nMeaning: every time the model gets a churner wrong,")
print(f"it counts as {weight:.1f} mistakes on the score card.")
print(f"The model has to work {weight:.1f}× harder to avoid missing churners.")

In [ ]:
# What happens with different imbalance levels?
print(f"{'Churn Rate':<12} {'Stayed':<10} {'Left':<10} {'Weight':<10} {'Interpretation'}")
print("-" * 70)

for churn_pct in [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]:
    n = 1000
    n_churn = int(n * churn_pct)
    test_labels = np.array([0] * (n - n_churn) + [1] * n_churn)
    w = compute_scale_pos_weight(test_labels)
    interp = f"Each missed churner counts as {w:.1f} mistakes" if w > 1 else "No adjustment needed"
    print(f"{churn_pct:<12.0%} {n - n_churn:<10} {n_churn:<10} {w:<10.2f} {interp}")

Notice: when churn rate hits 30% or above, we stop applying the weight (returns 1.0). At that point the dataset is balanced enough that the model doesn't need extra help.

## Part 3: XGBoost Hyperparameters

### First: What Is a Decision Tree?

Imagine playing 20 Questions to guess if a customer will leave:

- "Is their contract month-to-month?" → Yes
- "Have they been a customer for less than 6 months?" → Yes
- "Did they call support more than 3 times?" → Yes
- → "They're probably leaving."

That's a decision tree — a chain of yes/no questions that leads to a prediction. Each question splits customers into two groups. After enough questions, you end up with small groups that are mostly churners or mostly stayers.

A single tree isn't great. It might only get 60% of predictions right. But here's the trick...

### What Is XGBoost? (The Team of Trees)

XGBoost builds a TEAM of trees (usually 100-300). But they don't work independently — they learn from each other's mistakes:

1. **Tree 1** makes predictions. Gets some wrong.
2. **Tree 2** looks at what Tree 1 got wrong. Focuses specifically on those customers. Makes its own predictions.
3. **Tree 3** looks at what Trees 1+2 STILL get wrong. Focuses on the hardest cases.
4. ...repeat 100+ times.

The final prediction is the combined vote of all trees. Each individual tree is mediocre, but together they're remarkably accurate — like a committee where each member brings a different perspective.

### Hyperparameters: The Settings Knobs

Each hyperparameter controls HOW this team of trees works. Think of tuning a recipe — you adjust oven temperature, cook time, and ingredient ratios until the dish comes out right. The tuner tries many combinations automatically and picks whichever produces the best predictions on the validation set.

Here are the settings we tune:

In [ ]:
print("XGBoost Hyperparameter Ranges")
print("=" * 70)
for name, config in HYPERPARAMETER_RANGES.items():
    print(f"\n  {name} (range: {config['min']} to {config['max']})")
    print(f"    {config['description']}")

print("\n" + "=" * 70)
print("\nIn plain English:")
print("  - max_depth: How long can each chain of questions be?")
print("    Too short (3) = misses subtle patterns.")
print("    Too long (10) = memorizes individual customers instead of learning general rules.")
print("")
print("  - eta: How much does each new tree change the overall prediction?")
print("    Low (0.01) = cautious, needs many trees, but generalizes well.")
print("    High (0.3) = aggressive, fewer trees needed, but might overreact to noise.")
print("")
print("  - subsample: What fraction of customers does each tree get to see?")
print("    If every tree sees all customers, they all learn the same thing.")
print("    Showing each tree only 70% of customers forces diversity — different trees")
print("    learn different patterns, making the team stronger overall.")

## Part 4: From Predictions to Risk Tiers

After training, the model produces a probability for each customer (0.0 to 1.0). But business users don't want to interpret `0.73` — they want to know: **"Should I act NOW or later?"**

Risk tiers convert the raw number into an actionable label:

| Tier | Threshold | Action |
|------|-----------|--------|
| **High** | >= 0.7 | Intervene NOW — this customer is very likely leaving |
| **Medium** | 0.4 to 0.7 | Monitor closely — at risk, consider reaching out |
| **Low** | < 0.4 | Safe for now — no immediate action needed |

In [ ]:
# Demonstrate risk tier assignment
test_probs = [0.92, 0.85, 0.70, 0.55, 0.40, 0.39, 0.20, 0.05]

print(f"{'Probability':<14} {'Risk Tier':<12} {'Action'}")
print("-" * 55)
for p in test_probs:
    tier = assign_risk_tier(p)
    actions = {
        "high": "Intervene immediately",
        "medium": "Monitor, consider outreach",
        "low": "No action needed",
    }
    print(f"{p:<14.2f} {tier:<12} {actions[tier]}")

## Part 5: SHAP Explanations — "Show Your Work"

When the model says "Customer X: 85% chance of leaving," the client asks: **WHY?**

SHAP (SHapley Additive exPlanations) answers with **numbers, not words**. For each prediction, it computes how much each feature pushed the probability up or down:

- `contract_type (+0.23)` → this feature pushed churn probability UP by 23 percentage points
- `tenure_months (+0.15)` → this pushed it UP by 15 points
- `partner_status (-0.08)` → this pushed it DOWN by 8 points

**Important:** SHAP has no intelligence. It doesn't know WHY month-to-month contracts are risky. It doesn't generate explanations like "this customer has no commitment." It just does math — measuring each feature's contribution to the final number.

The **human-readable paragraph** ("This customer has no long-term commitment...") comes from a completely different step: the LLM narrative generator (Chapter 7). That's where Claude reads these raw SHAP numbers and writes English for business users.

For now, we just extract the **top 3 features by magnitude** — the three factors that had the biggest influence on this prediction, regardless of direction.

In [ ]:
# Simulate SHAP values for a high-risk customer
feature_names = ["tenure_months", "monthly_charges", "total_charges",
                  "contract_type", "support_tickets", "interaction_charges_x_tenure"]

# Customer with high churn probability
shap_high_risk = np.array([0.15, 0.08, -0.03, 0.23, 0.18, 0.05])

reasons = extract_top_reasons(shap_high_risk, feature_names, top_n=3)

print("High-risk customer — Top 3 SHAP factors (raw numbers, not explanations):")
print("=" * 55)
for i, reason in enumerate(reasons, 1):
    print(f"  {i}. {reason}")

print("\nWhat these numbers mean (a human or LLM would write this, not SHAP):")
print("  contract_type (+0.23) → being on month-to-month is the biggest risk factor")
print("  support_tickets (+0.18) → frequent support calls signal frustration")
print("  tenure_months (+0.15) → very short tenure, hasn't built loyalty")
print("\n  (In the final output, Claude writes these as plain-English paragraphs — see Chapter 7)")

In [ ]:
# Low-risk customer — notice the negative (protective) factors
shap_low_risk = np.array([-0.22, -0.05, -0.12, -0.31, 0.02, -0.15])

reasons = extract_top_reasons(shap_low_risk, feature_names, top_n=3)

print("Low-risk customer — Top 3 SHAP factors:")
print("=" * 45)
for i, reason in enumerate(reasons, 1):
    print(f"  {i}. {reason}")

print("\nWhat a human (or Claude) would interpret these as:")
print("  contract_type (-0.31) → long-term contract locks them in")
print("  tenure_months (-0.22) → been a customer for years")
print("  interaction (-0.15) → high lifetime value")
print("\n  Negative numbers = pushed AWAY from churn = protective factors")

## Part 6: The Final Deliverable

Everything comes together in the output CSV — what the client actually receives. One row per customer with:
- Who they are (`customer_id`)
- How likely they are to leave (`churn_probability`)
- The human-friendly label (`risk_tier`)
- The top 3 SHAP factors (`top_3_reasons`) — raw numbers showing which features mattered most
- A plain-English explanation (`narrative_explanation`) — this comes from the LLM step (Chapter 7). Without it, this column shows "N/A"

For now, we build the deliverable WITHOUT narratives — just the numbers. The LLM layer is optional and added later.

In [ ]:
# Build a sample output (WITHOUT LLM narratives — that comes in Chapter 7)
customer_ids = ["CUST_001", "CUST_002", "CUST_003", "CUST_004"]
probabilities = np.array([0.87, 0.55, 0.34, 0.92])
shap_matrix = np.array([
    [0.15, 0.08, -0.03, 0.23, 0.18, 0.05],   # high risk
    [0.10, 0.12, -0.05, 0.08, 0.06, -0.03],   # medium risk
    [-0.22, -0.05, -0.12, -0.31, 0.02, -0.15], # low risk
    [0.20, 0.15, 0.05, 0.25, 0.22, 0.10],     # very high risk
])

# No narratives yet — those get filled in by the LLM step (Chapter 7)
# Without the LLM, this column is just "N/A"
output_df = format_predictions(
    customer_ids, probabilities, shap_matrix, feature_names
)

print("Client Deliverable (predictions.csv):")
print("=" * 80)
for _, row in output_df.iterrows():
    print(f"\n  {row['customer_id']} | prob={row['churn_probability']} | tier={row['risk_tier']}")
    print(f"    SHAP factors: {row['top_3_reasons']}")
    print(f"    Narrative: {row['narrative_explanation']}")

print("\n" + "-" * 80)
print("Notice: narrative_explanation is 'N/A' for all customers.")
print("The LLM fills this in later (Chapter 7). The pipeline works fine without it —")
print("clients still get the probability, tier, and raw SHAP factors.")

## Key Takeaways

1. **Stratified splitting** preserves class balance across train/val/test — each group looks like the whole
2. **scale_pos_weight** prevents lazy predictions by making the model pay extra attention to the rare class
3. **Risk tiers** convert probabilities into actionable labels (high/medium/low)
4. **SHAP** shows the model's work — the top factors pushing each prediction up or down
5. **The output CSV** combines everything into one client-ready deliverable

Next: Chapter 5 covers the evaluation gate — how we decide if a model is good enough to serve to clients.

---

*Source code: `src/churn_pipeline/steps/training.py`, `src/churn_pipeline/steps/scoring.py`*  
*Tests: `tests/property/test_training.py`, `tests/property/test_scoring.py`*  
*Series: [Build with AWS](https://buildwithaws.substack.com/)*